### Imports

In [3]:
import decimal
import re

import pandas as pd
import sqlglot
import sqlglot.expressions as exp
from sqlglot.errors import ParseError, TokenError

### Load Dataset

In [4]:
# df = pd.read_csv('data/SQL_injection_Dataset.csv')
# df = pd.read_csv('data/clean_sql_dataset.csv')
df = pd.read_csv('data/test.csv')

In [5]:
# list of dialects was gotten from the following list, then ordered by usage/relevance
# all_dialects = list(sqlglot.Dialect.classes.keys())

# this ordering is AI generated, idea is to prioritize the most common/relevant dialects first, just to speed up the validation process a bit
dialects_to_try = [
    # Tier 1: The Default
    None,

    # Tier 2: The "Big Four" Web & Enterprise DBs (Highest SQLi probability)
    'mysql',
    'postgres',
    'oracle',
    'tsql',
    'sqlite',

    # Tier 3: Modern Cloud & Enterprise Data Warehouses
    'bigquery',
    'snowflake',
    'redshift',
    'databricks',

    # Tier 4: Big Data & Distributed SQL (Less common for direct web injection)
    'hive',
    'spark',
    'spark2',
    'presto',
    'trino',
    'athena',

    # Tier 5: OLAP, Search, and Niche/Specialized Engines
    'clickhouse',
    'duckdb',
    'teradata',
    'exasol',
    'druid',
    'dremio',
    'drill',
    'solr',
    'tableau',

    # Tier 6: Emerging or Highly Specific Dialects
    'dune',
    'prql',
    'doris',
    'fabric',
    'materialize',
    'risingwave',
    'singlestore',
    'starrocks'
]

In [6]:
# AI generated
def clean_waf_artifacts(payload):
    """
    Replaces known WAF evasion fuzzing clusters from the dataset
    with a safe dummy value so the rest of the AST can parse.
    """
    # Regex patterns targeting the exact garbage from your failing list
    # We replace them with '1' (a valid literal) so the UNION SELECT balances
    patterns_to_fix = [
        r'\+\\\.',      # Matches +\.
        r'\\\.\<\\',    # Matches \.<\
        r'%!\<@',       # Matches %!<@
        r'\\\<\\',      # Matches \<\
        r'\\#',         # Matches \#
        r'!\<@',        # Matches !<@
        r'\<@\.\.',     # Matches <@..
        r'\<@\.\$',     # Matches <@.$
        r'\$\s\.',      # Matches $ .
        r'\\\.+',       # Matches \.
        r'\"\"\"\"\<@'  # Matches """"<@
    ]

    for pattern in patterns_to_fix:
        payload = re.sub(pattern, ' 1 ', payload)

    # # Fix the weird isolated backslash in parentheses: * ( \ )
    # payload = payload.replace(r'* (  \  )', ' * ( 1 ) ')

    return payload

In [7]:
import json as _json

def extract_sql_features(payload):
    if not isinstance(payload, str):
        return {
            "is_valid_syntax": False,
            "winning_context_index": -1,
            "winning_dialect": None,
            "tables": "[]",
            "columns": "[]",
            "literal_types": "[]",
            "select_arm_widths": "[]",
            "node_set": "[]"
        }

    clean_payload = payload.strip('"').strip()
    clean_payload = clean_waf_artifacts(clean_payload)

    contexts = [
        f"{clean_payload}",
        f"SELECT * FROM users WHERE 1=1 {clean_payload}",
        f"SELECT * FROM users WHERE name = 'dummy{clean_payload}",
        f'SELECT * FROM users WHERE name = "dummy{clean_payload}',
        f"SELECT * FROM users WHERE 1=1 {clean_payload}'",
        f"SELECT * FROM users WHERE name = 'dummy{clean_payload}'",
        f"SELECT * FROM users WHERE (name = 'dummy{clean_payload}')",
        f"SELECT * FROM users WHERE (1=1 {clean_payload})",
        f"SELECT * FROM users; {clean_payload}",
        f"SELECT * FROM users WHERE name = 'dummy'; {clean_payload}"
    ]

    for dialect in dialects_to_try:
        for index, query in enumerate(contexts):
            try:
                parsed_statements = sqlglot.parse(query, read=dialect)

                tables = []
                # Fix 3: collect columns and their first seen literal type to get a sense of the schema and data shape, since we don't have access to the real DB schema or data
                col_type_map = {}  # column_name -> first seen literal type
                literal_types = []
                select_arm_widths = []
                node_set = set()

                for statement in parsed_statements:
                    if statement is None:
                        continue

                    for table in statement.find_all(exp.Table):
                        if table.name and table.name not in tables:
                            tables.append(table.name)

                    for literal in statement.find_all(exp.Literal):
                        if literal.args.get("is_string"):
                            literal_types.append("TEXT")
                        else:
                            literal_types.append("INTEGER")

                    for column in statement.find_all(exp.Column):
                        if column.name and column.name not in col_type_map:
                            # pair each new column with the next available literal type, defaulting to TEXT if none left
                            idx = len(col_type_map)
                            col_type_map[column.name] = (
                                literal_types[idx] if idx < len(literal_types) else "TEXT"
                            )

                    for select in statement.find_all(exp.Select):
                        if select.parent and type(select.parent) is exp.Union:
                            select_arm_widths.append(len(select.expressions))

                    for node in statement.walk():
                        node_set.add(type(node).__name__)

                columns = list(col_type_map.keys())
                col_literal_types = list(col_type_map.values())

                # Fix 1: return int(-1 sentinel) not None so pandas keeps the column as integer dtype (no float conversion)
                # Fix 2: json.dumps() so load_profiles can parse with json.loads() instead of getting Python repr like "['users']"
                return {
                    "is_valid_syntax": True,
                    "winning_context_index": int(index),
                    "winning_dialect": dialect if dialect else "default",
                    "tables": _json.dumps(tables),
                    "columns": _json.dumps(columns),
                    "literal_types": _json.dumps(col_literal_types),
                    "select_arm_widths": _json.dumps(select_arm_widths),
                    "node_set": _json.dumps(sorted(node_set))
                }

            except (ParseError, TokenError, ValueError, decimal.InvalidOperation):
                continue
            except Exception:
                continue

    # Fix 1: -1 instead of None keeps the column as int64 in pandas
    # Fix 2: '[]' strings so json.loads works in load_profiles
    return {
        "is_valid_syntax": False,
        "winning_context_index": -1,
        "winning_dialect": None,
        "tables": "[]",
        "columns": "[]",
        "literal_types": "[]",
        "select_arm_widths": "[]",
        "node_set": "[]"
    }


In [8]:
features_df = df['Query'].apply(extract_sql_features).apply(pd.Series)
df = pd.concat([df, features_df], axis=1)

In [9]:
print("Syntax Validation Summary, number of invalid within Label=1:")
print(df[df['Label'] == 1]['is_valid_syntax'].value_counts()) # 3070

Syntax Validation Summary, number of invalid within Label=1:
is_valid_syntax
True     106
False     13
Name: count, dtype: int64


In [10]:
# df.to_csv('data/SQL_injection_Dataset_Feature_Extraction_Results.csv', index=False)
# df.to_csv('data/clean_sql_dataset_Feature_Extraction_Results.csv', index=False)
df.to_csv('data/test_Feature_Extraction_Results.csv', index=False)


In [11]:
df.shape[0]

119

### **Overview**
This is a SQL injection detection system. It's different from other SQL injection detectors because
it looks at the behavior of the query, not the just what it looks like. It safely runs suspicious queries
inside a temporary, in-memory sqlite3 database to see if they actually act like an attack.

### **What It Does**
This SQL injection detection system runs suspicious queries in a controlled environment to see if
they behave like SQL injections. Here's how it does this.

1. AST Profiling: It starts by using a pre-computed abstract syntax tree (AST) profile to
    understand the structure of the query. It looks at things like the tables it's using and the way
    it is built.
2. Dynamic Database Creation: It creates a temporary sqlite3 database. It's designed to exactly
    fit the structure of the query it's running.
3. Canary Traps: It fills the database with special “trap” data. It's data that should not be
    returned unless something malicious is happening.
4. Execution & Monitoring: It runs the query and monitors it to see if it behaves suspiciously.
    The system flags the query if:
       o A UNION-based attack exposes any hidden “canary” data
       o A condition like OR 1=1, which forces all rows, including trap data, to be returned
       o Small changes in results, which indicates blind SQL injection techniques
5. Static Fallback: If the query is simply too broken to be executed, a strict set of regex-based
    checks are performed to catch obvious injection patterns.

### **Detection Categories**
The system groups any detected issues into a set of clear categories to make it easier to understand
what type of behavior was identified and why.

- union_based
- tautology
- blind_boolean
- blind_time
- stacked_queries
- comment_obfuscation
- encoding_obfuscation
- nested_injection

### **Command-Line Usage**
Use the test_sandbox.py script to test how the sandbox performs on data. The script can handle a
single CSV file that contains both original queries with labels and AST profile information.

**Standard Evaluation:**
```
python test_sandbox.py path/to/SQL_injection_Dataset_Feature_Extraction_Results.csv
```

**Limit Execution (evaluate only the first N rows):**
```
python test_sandbox.py path/to/SQL_injection_Dataset_Feature_Extraction_Results.csv --limit
2000
```

**Debug Mode (print False Positives and False Negatives):**
```
python test_sandbox.py path/to/SQL_injection_Dataset_Feature_Extraction_Results.csv --show-
fp --show-fn
```

**Adjust Debug Sample Size (default is 20):**
```
python test_sandbox.py path/to/SQL_injection_Dataset_Feature_Extraction_Results.csv --show-
fp --show-fn --sample 50
```


